# Payment Reconciliation Automation

> Synthetic reconstruction of a real-world payment reconciliation workflow.  
> No proprietary company data, code, credentials, or confidential business logic is included.

---

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    DOWNLOAD_DIR,
    UPLOAD_DIR,
    OUTPUT_DIR,
    BANK_A_FILE_PREFIX,
    BANK_B_FILE_PREFIX,
    API_ENDPOINT,
    INGESTION_TOKEN,
    INGESTION_PARENT_DIR,
)

from src.utils import get_source_files
from src.bank_a import process_bank_a
from src.bank_b import process_bank_b
from src.export import export_results
from src.ingestion import upload_file

---
## 2. Prepare Input Files
The workflow processes transaction statements received from multiple banking sources.

In [ ]:
bank_a_files = get_source_files(
    DOWNLOAD_DIR,
    BANK_A_FILE_PREFIX,
)

bank_b_files = get_source_files(
    DOWNLOAD_DIR,
    BANK_B_FILE_PREFIX,
)

print(f"Bank A files: {len(bank_a_files)}")
print(f"Bank B files: {len(bank_b_files)}")

---

## 3. Process Bank A Statements
Bank-specific processing includes:

- detecting the actual data header
- standardizing columns
- cleaning dates and amounts
- extracting transaction references
- classifying transactions

In [ ]:
bank_a_results = process_bank_a(
    bank_a_files
)

In [ ]:
# Inspect the output
for transaction_type, df in bank_a_results.items():
    print(
        f"{transaction_type}: "
        f"{len(df):,} rows"
    )

---

## 4. Process Bank B Statements
Bank B has a different statement structure and transaction-detail format, so it uses a separate processing module.

In [ ]:
bank_b_results = process_bank_b(
    bank_b_files
)

In [ ]:
# Inspect the output
for transaction_type, df in bank_b_results.items():
    print(
        f"{transaction_type}: "
        f"{len(df):,} rows"
    )

---

## 5. Export Processed Data
Processed datasets are exported as standardized CSV files.

In [ ]:
bank_a_outputs = export_results(
    results=bank_a_results,
    output_dir=OUTPUT_DIR,
    file_prefix="BANK_A",
)

In [ ]:
bank_b_outputs = export_results(
    results=bank_b_results,
    output_dir=OUTPUT_DIR,
    file_prefix="BANK_B",
)

In [ ]:
# Review generated files:

print("Bank A outputs:")
for file in bank_a_outputs:
    print(file)

print("\nBank B outputs:")
for file in bank_b_outputs:
    print(file)

---

## 6. Upload to Data Ingestion Layer
In production, the processed files can be uploaded to a downstream data platform through an ingestion API.

For this public repository, the API configuration is synthetic and should not contain real credentials.

In [ ]:
for file in bank_a_outputs + bank_b_outputs:

    upload_file(
        api_endpoint=API_ENDPOINT,
        ingestion_token=INGESTION_TOKEN,
        file_path=file,
        parent_dir=INGESTION_PARENT_DIR,
    )

> The upload step is included to demonstrate the production-style architecture.  
> The public repository does not connect to any real company or banking system.


---

## 7. Next Step: SQL Reconciliation

After ingestion, the processed transaction data is available in the downstream data layer.

The reconciliation itself is performed using SQL.

```text
Source Bank Statements
        ↓
Python Processing
        ↓
Standardized Transaction Files
        ↓
Data Ingestion API
        ↓
SQL Staging Tables
        ↓
SQL Reconciliation
        ↓
Variance Classification
        ↓
RCA / Manual Investigation
```

The SQL layer handles:

- transaction matching
- amount validation
- missing transactions
- duplicate detection
- reconciliation status
- data quality checks
- variance analysis

See:

```text
sql/
├── 01_staging.sql
├── 02_reconciliation.sql
└── 03_data_quality.sql
```

---

## Workflow Summary


    Payment Reconciliation Workflow

    1. Read source statements
    2. Clean and standardize data
    3. Extract transaction references
    4. Classify transactions
    5. Export standardized datasets
    6. Upload to ingestion layer
    7. Reconcile transactions using SQL
    8. Analyze reconciliation variances
